## Imports and basic definitions

In this first cell we collect all external libraries used throughout the project and define
basic single-spin operators.

We work with spin-1/2 systems, so the local Hilbert space is two-dimensional.
Here we explicitly define the Pauli matrices, ladder operators, and the identity.
These will be used to construct many-body operators via tensor products.

We also define a helper function `kron_n`, which computes the Kronecker product
of an arbitrary number of matrices. This is convenient for building operators
acting on multi-spin systems.

In [36]:
import numpy as np
import scipy.linalg as la
import matplotlib.pyplot as plt
from tqdm import tqdm
from scipy.optimize import minimize
from scipy.linalg import norm
from scipy.linalg import expm, sinm, cosm
import random
import cvxpy as cp
import itertools
import random
from scipy.linalg import eigh


sx = np.array([[0, 1], [1, 0]])  
sz = np.array([[1, 0], [0, -1]])  
sy = np.array([[0, -1j], [1j, 0]])  
identity = np.eye(2)  
I = np.eye(2)
s_plus = np.array([[0, 1], [0, 0]])
s_minus = np.array([[0, 0], [1, 0]])

def kron_n(*matrices):
    result = np.array([[1.0]])  
    for matrix in matrices:
        result = np.kron(result, matrix)
    return result

## Model Hamiltonian

We consider a one-dimensional spin-1/2 chain described by the following local Hamiltonian:

$$
\begin{aligned}
H ={}& -J_1 \sum_i \sigma^x_i \sigma^x_{i+1}
      -J_2 \sum_i \sigma^x_i \sigma^x_{i+2}
      -B   \sum_i \sigma^z_i \\
     & -G   \sum_i \sigma^x_i
      -J_y \sum_i \sigma^y_i \sigma^y_{i+1}
      -\Delta \sum_i \sigma^z_i \sigma^z_{i+1}.
\end{aligned}
$$

This Hamiltonian includes nearest- and next-nearest-neighbor interactions,
transverse and longitudinal fields, and anisotropic spin couplings.
By tuning the parameters, it interpolates between several paradigmatic
many-body models.



By choosing specific values of the coupling constants, the Hamiltonian reduces
to well-known many-body models:

- **Axial Next-Nearest Neighbor Ising (ANNNI) model**  
  $$
  G = J_y = \Delta = 0
  $$
  This model features competing nearest- and next-nearest-neighbor Ising interactions
  in a transverse field.

- **XXZ Heisenberg model with transverse field**  
  $$
  G = J_2 = 0, \quad J_y = J_1
  $$
  This corresponds to an anisotropic Heisenberg chain with a longitudinal field.

- **Mixed Field Ising (MFI) model**  
  $$
  J_1 = J_2 = J_y = 0
  $$
  A purely field-driven Ising model with both transverse and longitudinal components.

These limits allow us to study a broad range of physical behaviors within a single,
unified framework.


In [48]:
def build_hamiltonian(
    L,
    J1=0.0,      # XX nearest neighbor
    J2=0.0,      # XX next-nearest neighbor
    Jy=0.0,      # YY nearest neighbor
    Delta=0.0,   # ZZ nearest neighbor
    B=0.0,       # longitudinal field (Z)
    G=0.0        # transverse field (X)
):
    """
    Generic spin-1/2 Hamiltonian with OPEN boundary conditions:

    H = - J1 sum_i X_i X_{i+1}
        - J2 sum_i X_i X_{i+2}
        - Jy sum_i Y_i Y_{i+1}
        - Delta sum_i Z_i Z_{i+1}
        - B sum_i Z_i
        - G sum_i X_i
    """
    H = np.zeros((2**L, 2**L), dtype=complex)

    # ----- nearest-neighbor terms -----
    for j in range(L - 1):

        if J1 != 0.0:
            ops = [identity] * L
            ops[j], ops[j+1] = sx, sx
            H -= J1 * kron_n(*ops)

        if Jy != 0.0:
            ops = [identity] * L
            ops[j], ops[j+1] = sy, sy
            H -= Jy * kron_n(*ops)

        if Delta != 0.0:
            ops = [identity] * L
            ops[j], ops[j+1] = sz, sz
            H -= Delta * kron_n(*ops)

    # ----- next-nearest-neighbor XX -----
    for j in range(L - 2):

        if J2 != 0.0:
            ops = [identity] * L
            ops[j], ops[j+2] = sx, sx
            H -= J2 * kron_n(*ops)

    # ----- local fields -----
    for j in range(L):

        if B != 0.0:
            ops = [identity] * L
            ops[j] = sz
            H -= B * kron_n(*ops)

        if G != 0.0:
            ops = [identity] * L
            ops[j] = sx
            H -= G * kron_n(*ops)

    return H


## Pauli-string operators and expectation values

In this section we define utilities to work with *Pauli strings*, i.e. tensor products
of single-site Pauli operators of the form

$$
P = \sigma^{\alpha_1}_1 \otimes \sigma^{\alpha_2}_2 \otimes \cdots \otimes \sigma^{\alpha_L}_L,
\qquad \alpha_i \in \{I, X, Y, Z\}.
$$

In [49]:
def pauli_operator(pauli_string):
    """
    Construct a many-body Pauli operator from a string of characters
    in {'I', 'X', 'Y', 'Z'}.
    """
    op_list = [identity] * len(pauli_string)

    for i, char in enumerate(pauli_string):
        if char == 'X':
            op_list[i] = sx
        elif char == 'Y':
            op_list[i] = sy
        elif char == 'Z':
            op_list[i] = sz
        elif char == 'I':
            op_list[i] = identity

    return kron_n(*op_list)


def pauli_expectation(rho, pauli_string, tol=1e-8):
    """
    Compute the expectation value of a Pauli string with respect to
    a density matrix rho.
    """
    P = pauli_operator(pauli_string)
    exp_value = np.trace(rho @ P)

    if np.abs(exp_value.imag) > tol:
        print(
            f"[Warning] Large imaginary part detected: "
            f"{exp_value.imag} for Pauli string {pauli_string}"
        )

    return exp_value.real


def random_pauli_strings(L, Ns, seed=None):
    """
    Generate Ns random Pauli strings of length L (with repetitions allowed).
    """
    if seed is not None:
        random.seed(seed)

    pauli_strings = []

    for _ in range(Ns):
        string = ''.join(random.choices(['I', 'X', 'Y', 'Z'], k=L))
        pauli_strings.append(string)

    return pauli_strings


def unique_random_pauli_strings(L, Ns, seed=None):
    """
    Generate Ns unique Pauli strings of length L.
    """
    if seed is not None:
        random.seed(seed)

    paulis = ['I', 'X', 'Y', 'Z']
    all_strings = [''.join(p) for p in itertools.product(paulis, repeat=L)]

    if Ns > len(all_strings):
        raise ValueError(
            f"Requested {Ns} strings, but only {len(all_strings)} unique strings available."
        )

    return random.sample(all_strings, Ns)


def pauli_expectations(rho, strings):
    """
    Compute expectation values for a list of Pauli strings.
    """
    values = []

    for s in strings:
        values.append(pauli_expectation(rho, s))

    return np.array(values)


**Hierarchical sampling of Pauli observables**

To generate the set of measured observables, we use a *hierarchical sampling*
of Pauli strings, prioritizing few-body operators.

The procedure is as follows:
- All Pauli strings are grouped by their **body order** (number of non-identity sites)
- One-body operators are sampled first, followed by two-body operators, and so on
- Within each body-order sector, Pauli strings are **randomly shuffled**
- The process stops once the desired number $ K = |\mathcal{I}| $ of observables
  is reached

This strategy reflects physically motivated measurement schemes and improves
the efficiency of the certification protocol compared to fully random sampling.

In [50]:
import itertools
import random

def unique_random_pauli_strings_fewbodies(L, Ns, seed=None):
    """
    Generate up to Ns unique Pauli strings prioritizing few-body terms:
    1-body first, then 2-body, then 3-body, etc.
    Randomized within each body order, but ordered by increasing body size.
    """
    if seed is not None:
        random.seed(seed)

    ordered_strings = []

    for n in range(1, L + 1):
        n_body_strings = []

        for positions in itertools.combinations(range(L), n):
            for ops in itertools.product(['X', 'Y', 'Z'], repeat=n):
                s = ['I'] * L
                for pos, op in zip(positions, ops):
                    s[pos] = op
                n_body_strings.append(''.join(s))

        random.shuffle(n_body_strings)
        ordered_strings.extend(n_body_strings)

        if len(ordered_strings) >= Ns:
            break

    if Ns > len(ordered_strings):
        raise ValueError(
            f"Requested {Ns} unique Pauli strings, but only "
            f"{len(ordered_strings)} available for L={L}."
        )

    return ordered_strings[:Ns]


**Statistical post-processing** of numerical experiments  
   We compute medians and interquartile ranges of data obtained from multiple
   realizations and stored on disk.

In [51]:
def compute_quartile_stats(prefix, Ns_list, num_realizations):
    """
    Load data from text files and compute median and interquartile ranges.

    Files are assumed to be named:
    {prefix}_{Ns}paulis_real{num_realizations}.txt
    """
    medians, lower, upper = [], [], []

    for Ns in Ns_list:
        data = np.loadtxt(f"{prefix}_{Ns}paulis_real{num_realizations}.txt")
        med = np.median(data)
        q25, q75 = np.percentile(data, [25, 75])

        medians.append(med)
        lower.append(med - q25)
        upper.append(q75 - med)

    return np.array(medians), np.array(lower), np.array(upper)

**Optimal unitary alignment between states and Hamiltonians**  
   Given a density matrix and a Hamiltonian, we construct the unitary that aligns
   the eigenbasis of the state (ordered by decreasing eigenvalues) with the
   eigenbasis of the Hamiltonian (ordered by increasing energies).
   

In [52]:
def optimal_unitary(rho, H):
    """
    Construct the unitary U that aligns the eigenbasis of rho (descending eigenvalues)
    with the eigenbasis of H (ascending eigenvalues).
    """
    # Diagonalize rho
    r_evals, r_evecs = np.linalg.eigh(rho)
    r_sort_idx = np.argsort(-r_evals)
    r_evecs = r_evecs[:, r_sort_idx]

    # Diagonalize H
    H_evals, H_evecs = np.linalg.eigh(H)
    H_sort_idx = np.argsort(H_evals)
    H_evecs = H_evecs[:, H_sort_idx]

    # U = sum_j |E_j><r_j|
    U = H_evecs @ r_evecs.conj().T
    return U

## SDP protocol

**Semidefinite programs (SDPs) for density-matrix reconstruction**  
   We reconstruct density matrices consistent with a given set of Pauli
   expectation values by solving convex optimization problems.
   Two variants are implemented:
   - Energy-based minimization
   - Maximum-mixedness (minimum purity) reconstruction

In [53]:
def sdp_minimization(L, pauli_strings, pauli_values, U, H):
    """
    SDP to reconstruct a density matrix consistent with Pauli expectations,
    minimizing an energy-based cost function.
    """
    rho = cp.Variable((2**L, 2**L), hermitian=True)

    # Objective: Tr[rho H] - Tr[U† rho U H]
    objective = cp.Minimize(
        cp.real(cp.trace((H - U.conj().T @ H @ U) @ rho))
    )

    constraints = [
        cp.trace(rho) == 1,
        rho >> 0,
    ]

    for i, pauli_string in enumerate(pauli_strings):
        Oi = pauli_operator(pauli_string)
        oi = cp.real(pauli_values[i])
        constraints.append(cp.real(cp.trace(rho @ Oi)) == oi)

    try:
        problem = cp.Problem(objective, constraints)
        problem.solve(solver=cp.SCS, verbose=False)

        if problem.status not in ["optimal", "optimal_inaccurate"]:
            print(f"[SDP Warning] Problem status: {problem.status}")
            return np.inf, None

        return problem.value, rho.value

    except Exception as e:
        print(f"[SDP Error] Solver failed: {e}")
        return np.inf, None


In [54]:
def sdp_most_mixed_state(L, pauli_strings, pauli_values, solver=cp.SCS):
    """
    Reconstruct the most mixed (minimum purity) density matrix consistent
    with given Pauli expectation values.
    """
    rho = cp.Variable((2**L, 2**L), hermitian=True)

    purity = cp.sum_squares(cp.abs(rho))
    objective = cp.Minimize(cp.real(purity))

    constraints = [
        cp.trace(rho) == 1,
        rho >> 0,
    ]

    for i, pauli_string in enumerate(pauli_strings):
        Oi = pauli_operator(pauli_string)
        oi = float(np.real(pauli_values[i]))
        constraints.append(cp.real(cp.trace(rho @ Oi)) == oi)

    try:
        problem = cp.Problem(objective, constraints)
        problem.solve(solver=solver, verbose=False)

        if problem.status not in ["optimal", "optimal_inaccurate"]:
            print(f"[SDP Warning] status: {problem.status}")
            return None

        return rho.value

    except Exception as e:
        print(f"[SDP Error] {e}")
        return None


## Negative-temperature state

In this section we compute certified lower bounds on the ergotropy as a function
of the number $ K = |\mathcal{I}| $ of measured observables (random Pauli strings),
for a **negative-temperature Gibbs state** $ \beta = -1 $ of a five-qubit system.

We consider:
- A mixed-field Ising (MFI) Hamiltonian
- Hierarchical sampling of Pauli observables
- \(20\) independent realizations for each value of \(K\)

The plotted curve shows the **median** over realizations, with shaded regions
indicating the **interquartile range (IQR)**.
A dashed horizontal line marks the **true ergotropy**.


In [55]:
# System size
L = 5

# MFI Hamiltonian parameters
J1 = 0.0
J2 = 0.0
Jy = 0.0
Delta = 1.0
B = 0.5
G = 0.5

# Build Hamiltonian (general function)
H_mfi = build_hamiltonian(
    L,
    J1=J1,
    J2=J2,
    B=B,
    G=G,
    Jy=Jy,
    Delta=Delta
)

# Negative-temperature Gibbs state
beta_neg = -1.0
rho_tneg_unnorm = expm(-beta_neg * H_mfi)
rho_tneg = rho_tneg_unnorm / np.trace(rho_tneg_unnorm)

# True ergotropy
p_tneg, _ = np.linalg.eigh(rho_tneg)
e_tneg, _ = np.linalg.eigh(H_mfi)

p_sorted = np.sort(p_tneg)[::-1]
e_sorted = np.sort(e_tneg)

energy_init = np.trace(rho_tneg @ H_mfi)
energy_min = np.sum(p_sorted * e_sorted)

ergotropy_tneg = energy_init - energy_min

print(f"True ergotropy (MFI, negative temperature): {ergotropy_tneg:.6f}")


True ergotropy (MFI, negative temperature): 8.308158+0.000000j


**SDP-based certification protocol**

For each number of measured observables $K$:

1. Generate a random set of Pauli strings
2. Measure their expectation values on the true state
3. Reconstruct a feasible density matrix via a maximum-mixedness SDP
4. Compute the optimal unitary aligning the state with the Hamiltonian
5. Solve a second SDP to obtain a certified ergotropy lower bound

This procedure is repeated for 20 independent realizations.

In [56]:
Ns_list_tneg = list(range(1, 600, 20))
num_realizations = 20

for Ns in tqdm(Ns_list_tneg, desc="Negative-temperature MFI"):
    realization_values = []

    for realization in range(num_realizations):
        seed = 42 + realization

        pauli_strings = unique_random_pauli_strings_fewbodies(L, Ns, seed)
        pauli_vals = pauli_expectations(rho_tneg, pauli_strings)

        # First SDP: feasible state
        rho_feas = sdp_most_mixed_state(L, pauli_strings, pauli_vals)
        if rho_feas is None:
            continue

        # Optimal unitary
        U_opt = optimal_unitary(rho_feas, H_mfi)

        # Second SDP: ergotropy bound
        min_val, _ = sdp_minimization(
            L, pauli_strings, pauli_vals, U_opt, H_mfi
        )

        if np.isfinite(min_val):
            realization_values.append(min_val)

    realization_values = np.array(realization_values)

    # 🔑 filename compatible with compute_quartile_stats
    np.savetxt(
        f"tneg_MFI_{Ns}paulis_real{num_realizations}.txt",
        realization_values
    )

print("All realizations saved successfully.")


Negative-temperature MFI: 100%|██████████| 30/30 [13:09<00:00, 26.32s/it]

All realizations saved successfully.


In [57]:
median_tneg, lower_err_tneg, upper_err_tneg = compute_quartile_stats(
    prefix="tneg_MFI",
    Ns_list=Ns_list_tneg,
    num_realizations=num_realizations
)


**Plot: Certified ergotropy lower bounds**

We now plot the median certified ergotropy lower bound as a function of
the number of measured Pauli observables $ |\mathcal{I}| $.


In [113]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rc
import pylab

######### plot style (LaTeX) #########
rc('text', usetex=True)

fig_width_pt = 246.0
inches_per_pt = 1.0 / 72.27
fig_width = fig_width_pt * inches_per_pt
fig_size = [fig_width, 0.75 * fig_width]

params = {
    'backend': 'ps',
    'axes.labelsize': 10,
    'font.size': 9,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.usetex': True,
    'figure.figsize': fig_size
}
pylab.rcParams.update(params)

fig, ax = plt.subplots()

rc('lines', linewidth=1.2)
rc('axes', linewidth=1.2)

######### axes labels #########
#ax.set_xlabel(r'$|\mathcal{I}|$')
ax.set_xlabel(r'$K$')
#ax.set_ylabel(r'Ergotropy lower bound')
ax.set_ylabel(r'$\mathcal{E}_{\mathrm{LB}}$')


######### clip negatives #########
median_clipped = np.maximum(median_tneg, 0)

######### color #########
color = "#4C72B0"

######### IQR shading #########
ax.fill_between(
    Ns_list_tneg,
    np.maximum(median_tneg - lower_err_tneg, 0),
    np.maximum(median_tneg + upper_err_tneg, 0),
    color=color,
    alpha=0.25
)

######### median #########
ax.plot(
    Ns_list_tneg,
    median_clipped,
    color=color,
    marker='o',
    markersize=3,
    linestyle='-',
    #label=r'Median'
)

######### true ergotropy (same color) #########
ax.axhline(
    ergotropy_tneg,
    color=color,
    linestyle='--',
    linewidth=1.2,
    #label=r'True ergotropy'
)

######### zero line #########
ax.axhline(0, color='gray', linewidth=0.6)

######### grid #########
ax.grid(True, linestyle='--', linewidth=0.5)

######### legend #########
ax.legend(frameon=False)

######### save #########
filename = f"ergotropy_vs_K_tneg_MFI_L{L}_OBC"
plt.savefig(filename + ".pdf", dpi=300, bbox_inches='tight')
print(f"Saved figure as {filename}.pdf")

plt.show()


/Users/epagliaro/.julia/conda/3/aarch64/lib/python3.10/site-packages/matplotlib/cbook.py:1355: ComplexWarning: Casting complex values to real discards the imaginary part
  return np.asarray(x, float)
/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_67400/3439718319.py:80: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(frameon=False)


Saved figure as ergotropy_vs_K_tneg_MFI_L5_OBC.pdf


/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_67400/3439718319.py:87: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


## GHZ, W, product states- XXZ Hamiltonian

In this section we compute certified lower bounds on the ergotropy
as a function of the number $ K = |\mathcal{I}| $ of measured observables,
for three different L=5 states under the XXZ Hamiltonian ($J_1 = 1$, $J_y = 1$, $\Delta = 0.5$, $B = 0$):

- GHZ state ($|\mathrm{GHZ}\rangle$)  
- W state ($|W\rangle$)  
- Product state ($|0\dots0\rangle$)  

For each value of \(K\), we perform **20 independent realizations** with
hierarchical sampling of Pauli strings. The median is plotted as a line,
the shaded region represents the interquartile range (IQR), and
dashed horizontal lines mark the true ergotropy of each state.


In [227]:
# System size
L = 5

# XXZ Hamiltonian parameters
J1 = 1.0
J2 = 0.0
Jy = 1.0
Delta = 0.5
B = 0.0
G = 0.0

# Build Hamiltonian using general builder
H_xxz = build_hamiltonian(L, J1=J1, J2=J2, B=B, G=G, Jy=Jy, Delta=Delta)

# --- GHZ state ---
dim = 2**L
ghz_state = (np.zeros(dim))
ghz_state[0] = 1.0
ghz_state[-1] = 1.0
ghz_state /= np.sqrt(2)
rho_ghz = np.outer(ghz_state, ghz_state.conj())

# --- W state ---
def w_state(L):
    psi = np.zeros(2**L, dtype=complex)
    for i in range(L):
        psi[2**i] = 1
    psi /= np.sqrt(L)
    return psi
psi_W = w_state(L)
rho_W = np.outer(psi_W, psi_W.conj())

# --- Product state |0...0> ---
psi_prod = np.zeros(2**L, dtype=complex)
psi_prod[0] = 1.0
rho_prod = np.outer(psi_prod, psi_prod.conj())


**Hierarchical Pauli sampling and SDP procedure**

For each number of measured observables $K$:

1. Generate a hierarchical set of Pauli strings (few-body first) using `unique_random_pauli_strings_fewbodies`
2. Measure expectation values on the true state
3. Reconstruct a feasible density matrix via maximum-mixedness SDP
4. Compute the optimal unitary aligning the state with the Hamiltonian
5. Solve a second SDP to obtain a certified ergotropy lower bound

This procedure is repeated for 20 independent realizations per value of $K$.


In [83]:
# Measurement sizes and realizations
Ns_list_xxz = list(range(1, 101, 5))  # stop at 100, step=5
num_realizations = 20

states = {
    "GHZ": rho_ghz,
    "W": rho_W,
    "Product": rho_prod
}

for name, rho in states.items():
    for Ns in tqdm(Ns_list_xxz, desc=f"Running {name} state reconstructions"):
        realization_values = []

        for realization in range(num_realizations):
            seed = 42 + realization
            pauli_strings = unique_random_pauli_strings_fewbodies(L, Ns, seed)

            # Step 1: measurement expectations
            values = pauli_expectations(rho, pauli_strings)

            # Step 2: maximum-mixedness SDP
            rho_feas = sdp_most_mixed_state(L, pauli_strings, values)
            if rho_feas is None:
                continue

            # Step 3: optimal unitary
            U_opt = optimal_unitary(rho_feas, H_xxz)

            # Step 4: ergotropy SDP
            min_val, _ = sdp_minimization(L, pauli_strings, values, U_opt, H_xxz)
            if np.isfinite(min_val):
                realization_values.append(min_val)

        realization_values = np.array(realization_values)

        # Save each Ns realization vector
        np.savetxt(f"xxz_{name}_{Ns}paulis_real{num_realizations}.txt", realization_values)


Running GHZ state reconstructions:  20%|██        | 4/20 [00:10<00:43,  2.69s/it]/Users/epagliaro/.julia/conda/3/aarch64/lib/python3.10/site-packages/cvxpy/problems/problem.py:1504: UserWarning: Solution may be inaccurate. Try another solver, adjusting the solver settings, or solve with verbose=True for more information.
  warnings.warn(
Running Product state reconstructions: 100%|██████████| 20/20 [09:00<00:00, 27.02s/it]  


In [90]:
# Compute quartiles using existing function
median_ghz, lower_ghz, upper_ghz = compute_quartile_stats("xxz_GHZ", Ns_list_xxz, num_realizations)
median_w, lower_w, upper_w = compute_quartile_stats("xxz_W", Ns_list_xxz, num_realizations)
median_prod, lower_prod, upper_prod = compute_quartile_stats("xxz_Product", Ns_list_xxz, num_realizations)

# True ergotropy function
def true_ergotropy(rho, H):
    p, _ = np.linalg.eigh(rho)
    e, _ = np.linalg.eigh(H)
    p_sorted = np.sort(p)[::-1]
    e_sorted = np.sort(e)
    return np.real(np.trace(rho @ H)) - np.sum(p_sorted * e_sorted)

ergotropy_ghz = true_ergotropy(rho_ghz, H_xxz)
ergotropy_w = true_ergotropy(rho_W, H_xxz)
ergotropy_prod = true_ergotropy(rho_prod, H_xxz)


**Plot: Certified ergotropy lower bounds (XXZ states)**

Median certified ergotropy bounds are plotted as a function of the number of
measured Pauli observables $|\mathcal{I}|$ for the GHZ, W, and Product states.
Shaded regions indicate the interquartile range (IQR), and dashed horizontal
lines show the true ergotropy of each state.


In [228]:
import matplotlib.pyplot as plt
from matplotlib import rc
import pylab

# --- LaTeX style ---
rc('text', usetex=True)
fig_width_pt = 246.0
inches_per_pt = 1.0 / 72.27
fig_width = fig_width_pt * inches_per_pt
fig_size = [fig_width, 0.75 * fig_width]
params = {
    'backend': 'ps',
    'axes.labelsize': 10,
    'font.size': 9,
    'legend.fontsize': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.usetex': True,
    'figure.figsize': fig_size
}
pylab.rcParams.update(params)

fig, ax = plt.subplots()
rc('lines', linewidth=1.2)
rc('axes', linewidth=1.2)

# Colors
colors = {"GHZ": "#4C72B0", "W": "#B04C72", "Product": "#83AF67"}

# Clip medians & quartiles to nonnegative
med_ghz = np.maximum(median_ghz, 0)
low_ghz = np.maximum(median_ghz - lower_ghz, 0)
up_ghz = np.maximum(median_ghz + upper_ghz, 0)

med_w = np.maximum(median_w, 0)
low_w = np.maximum(median_w - lower_w, 0)
up_w = np.maximum(median_w + upper_w, 0)

med_prod = np.maximum(median_prod, 0)
low_prod = np.maximum(median_prod - lower_prod, 0)
up_prod = np.maximum(median_prod + upper_prod, 0)

# Plot IQR + median
ax.fill_between(Ns_list_xxz, low_ghz, up_ghz, color=colors["GHZ"], alpha=0.25)
ax.plot(Ns_list_xxz, med_ghz, color=colors["GHZ"], marker='o', markersize=3, linestyle='-')

ax.fill_between(Ns_list_xxz, low_w, up_w, color=colors["W"], alpha=0.25)
ax.plot(Ns_list_xxz, med_w, color=colors["W"], marker='o', markersize=3, linestyle='-')

ax.fill_between(Ns_list_xxz, low_prod, up_prod, color=colors["Product"], alpha=0.25)
ax.plot(Ns_list_xxz, med_prod, color=colors["Product"], marker='o', markersize=3, linestyle='-')

# True ergotropy dashed lines
ax.axhline(y=ergotropy_ghz, color=colors["GHZ"], linestyle='--', linewidth=1.2, label=r'$\mathrm{GHZ}$')
ax.axhline(y=ergotropy_w, color=colors["W"], linestyle='--', linewidth=1.2, label=r'$\mathrm{W}$')
ax.axhline(y=ergotropy_prod, color=colors["Product"], linestyle='--', linewidth=1.2, label=r'$\mathrm{Product}$')


# Zero line
ax.axhline(0, color='gray', linestyle='-', linewidth=0.6)

# Labels
#ax.set_xlabel(r"$|\mathcal{I}|$")
ax.set_xlabel(r"$K$")
ax.set_ylabel(r"$\mathcal{E}_{\mathrm{LB}}$")

# X-axis range and padding
padding = 0.05 * (max(Ns_list_xxz)-min(Ns_list_xxz))
plt.xlim([min(Ns_list_xxz)-padding, 100])

# Grid
ax.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.grid(True, which='minor', linestyle=':', linewidth=0.2)

# Legend
ax.legend(frameon=False)

# Save
filename = f"ergotropy_comparison_XXZ_L{L}_minpurity_OBC_K_legend.pdf"
plt.savefig(filename, dpi=300, bbox_inches='tight')
print(f"Saved figure as {filename}.pdf")

plt.show()


Saved figure as ergotropy_comparison_XXZ_L5_minpurity_OBC_K_legend.pdf.pdf


/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_67400/1102391282.py:83: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


## Probabilistic ergotropy lower bounds under finite sampling

We study the performance of the certification protocol in the presence of
finite-statistics measurement noise. We consider a five-qubit ANNNI model
with Hamiltonian parameters
$ J_1 = 1,\quad J_2 = -1,\quad B = 0.5$
and a pure target state given by a coherent superposition of extremal
Hamiltonian eigenstates,
$$
|\psi\rangle = \frac{1}{\sqrt{2}}\left(|\varepsilon_1\rangle + |\varepsilon_d\rangle\right),
\quad \rho = |\psi\rangle\langle\psi|.
$$

For each number $K = |\mathcal{I}|$ of measured Pauli observables, we
estimate probabilistic ergotropy lower bounds holding with confidence
level $1-\delta = 0.997$, using relaxed SDP constraints derived from
Hoeffding-type concentration bounds. 

Each data point corresponds to the median over 20 realizations obtained
via hierarchical sampling. Shaded regions indicate the interquartile
range across realizations. Different curves correspond to different
numbers of measurement shots per Pauli observable.


Relaxed SDP solvers:

In [96]:
import cvxpy as cp

def sdp_minimization_relaxed(L, pauli_strings, pauli_values, U, H, epsilon):
    """
    Second SDP: minimize ergotropy functional with relaxed Pauli constraints.
    """
    rho = cp.Variable((2**L, 2**L), hermitian=True)

    objective = cp.Minimize(
        cp.real(cp.trace((H - U.conj().T @ H @ U) @ rho))
    )

    constraints = [
        cp.trace(rho) == 1,
        rho >> 0,
    ]

    for i, pauli_string in enumerate(pauli_strings):
        Oi = pauli_operator(pauli_string)
        oi = cp.real(pauli_values[i])
        constraints += [
            cp.real(cp.trace(rho @ Oi)) >= oi - epsilon,
            cp.real(cp.trace(rho @ Oi)) <= oi + epsilon,
        ]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.SCS, verbose=False)

    if problem.status not in ["optimal", "optimal_inaccurate"]:
        return np.inf, None

    return problem.value, rho.value


def sdp_most_mixed_state_relaxed(L, pauli_strings, pauli_values, epsilon):
    """
    First SDP: find the most mixed feasible state consistent with data.
    """
    rho = cp.Variable((2**L, 2**L), hermitian=True)
    purity = cp.sum_squares(cp.abs(rho))

    objective = cp.Minimize(cp.real(purity))

    constraints = [cp.trace(rho) == 1, rho >> 0]

    for i, pauli_string in enumerate(pauli_strings):
        Oi = pauli_operator(pauli_string)
        oi = cp.real(pauli_values[i])
        constraints += [
            cp.real(cp.trace(rho @ Oi)) >= oi - epsilon,
            cp.real(cp.trace(rho @ Oi)) <= oi + epsilon,
        ]

    problem = cp.Problem(objective, constraints)
    problem.solve(solver=cp.SCS, verbose=False)

    if problem.status not in ["optimal", "optimal_inaccurate"]:
        return None

    return rho.value


Statistical error model:

In [97]:
def compute_epsilon(Ns, n_shots, delta=0.003):
    """
    Hoeffding-type confidence interval for K = Ns observables.
    """
    return np.sqrt(2 * np.log(2 * Ns / delta) / n_shots)


Finite-shot Pauli measurements:

In [98]:
def pauli_expectations_fake_shots(rho, pauli_list, n_shots, seed=None):
    """
    Simulate Pauli expectation values with finite measurement statistics.
    """
    rng = np.random.default_rng(seed)
    expectations = []

    for P in pauli_list:
        operator = pauli_operator(P)
        exp_val = np.real(np.trace(rho @ operator))

        prob_plus = np.clip((1 + exp_val) / 2, 0, 1)
        outcomes = np.where(rng.random(n_shots) < prob_plus, 1, -1)

        expectations.append(np.mean(outcomes))

    return np.array(expectations)


One realization of the protocol:

In [99]:
def run_single_realization(Ns, realization, L, rho, H, n_shots):
    """
    Run the full two-step probabilistic certification for one realization.
    """
    seed = 42 + realization
    epsilon = compute_epsilon(Ns, n_shots)

    pauli_strings = unique_random_pauli_strings_fewbodies(L, Ns, seed)
    values = pauli_expectations_fake_shots(rho, pauli_strings, n_shots, seed)

    rho_feas = sdp_most_mixed_state_relaxed(L, pauli_strings, values, epsilon)
    if rho_feas is None:
        return None

    U_opt = optimal_unitary(rho_feas, H)

    val, _ = sdp_minimization_relaxed(
        L, pauli_strings, values, U_opt, H, epsilon
    )

    return val if np.isfinite(val) else None


ANNNI Hamiltonian and target state:

In [100]:
from scipy.linalg import eigh
import numpy as np

# system size
L = 5
dim = 2**L

# ----- build ANNNI Hamiltonian (OPEN boundaries) -----
H_annni = build_hamiltonian(
    L,
    J1=1.0,     # nearest-neighbor XX
    J2=-1.0,    # next-nearest-neighbor XX
    B=0.5,      # longitudinal field
    Jy=0.0,
    Delta=0.0,
    G=0.0,
)

# ----- diagonalize Hamiltonian -----
e_vals, e_vecs = eigh(H_annni)

# extremal eigenstates
psi_min = e_vecs[:, 0]       # ground state
psi_max = e_vecs[:, -1]      # highest-energy state

# coherent superposition
psi_super = psi_min + psi_max
psi_super /= np.linalg.norm(psi_super)

# density matrix
rho_super = np.outer(psi_super, psi_super.conj())


Run simulations:

In [105]:
from joblib import Parallel, delayed
from tqdm import tqdm

# shot numbers
shot_list = [10000, 50000, 200000]

# number of measured observables
Ns_list = list(range(1, 1000, 40))

# realizations per point
num_realizations = 20

results_by_shots = {}

for n_shots in shot_list:
    print(f"\n=== Running simulations for n_shots = {n_shots} ===")

    all_vals = []

    for Ns in tqdm(Ns_list):
        vals = Parallel(n_jobs=-1)(
            delayed(run_single_realization)(
                Ns,
                realization,
                L,
                rho_super,
                H_annni,
                n_shots,
            )
            for realization in range(num_realizations)
        )

        # discard failed runs
        vals = np.array([v for v in vals if v is not None])
        all_vals.append(vals)

    results_by_shots[n_shots] = all_vals

print("All simulations completed.")



=== Running simulations for n_shots = 10000 ===


100%|██████████| 25/25 [05:43<00:00, 13.74s/it]



=== Running simulations for n_shots = 50000 ===


100%|██████████| 25/25 [06:44<00:00, 16.17s/it]



=== Running simulations for n_shots = 200000 ===


100%|██████████| 25/25 [08:51<00:00, 21.27s/it]

All simulations completed.


**Statistical post-processing**

For each value of $K$ and each shot number, we compute the median
ergotropy lower bound across realizations, together with the
interquartile range (IQR), which is displayed as a shaded region
in the final plot.


In [106]:
def compute_quartile_stats_from_array(data_list):
    """
    Compute median and IQR error bars from a list of arrays.
    """
    med, low, up = [], [], []

    for vals in data_list:
        q25, q75 = np.percentile(vals, [25, 75])
        m = np.median(vals)
        med.append(m)
        low.append(m - q25)
        up.append(q75 - m)

    return np.array(med), np.array(low), np.array(up)


In [108]:
def true_ergotropy_scalar(rho, H):
    """
    True ergotropy:
    E(rho, H) = Tr[rho H] - min_U Tr[U rho U† H]
              = Tr[rho H] - sum_j p_j^↓ E_j^↑
    """
    # Eigenvalues of rho
    p, _ = np.linalg.eigh(rho)
    p_sorted = np.sort(p)[::-1]

    # Eigenvalues of H
    E, _ = np.linalg.eigh(H)
    E_sorted = np.sort(E)

    # Initial energy
    E_init = np.real(np.trace(rho @ H))

    # Passive-state energy
    E_passive = np.sum(p_sorted * E_sorted)

    return E_init - E_passive


**Probabilistic ergotropy lower bounds**

The figure below shows the probabilistic ergotropy lower bound
$\mathcal{E}_{\mathrm{LB}}$ as a function of the number of measured
observables $K = |\mathcal{I}|$, for different numbers of measurement
shots per Pauli observable.

Shaded regions denote the interquartile range across realizations.


In [112]:
import matplotlib.pyplot as plt
import pylab
import numpy as np

# =======================
# Plot style (LaTeX + postdoc template)
# =======================
fig_width_pt = 246.0
inches_per_pt = 1.0 / 72.27
fig_width = fig_width_pt * inches_per_pt
fig_size = [fig_width, 0.75 * fig_width]

params = {
    "text.usetex": True,
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
    "xtick.labelsize": 9,
    "ytick.labelsize": 9,
    "figure.figsize": fig_size,
}
pylab.rcParams.update(params)

# =======================
# x-axis values
# =======================
Ns_list = list(range(1, 1000, 40))
x = np.array(Ns_list)

# =======================
# Colors (lighter → fewer shots)
# =======================
shot_color_map = {
    10000: "#4C72B0",
    50000: "#3C5B8C",
    200000: "#1E2D46",
}

# =======================
# True ergotropy
# =======================
erg_true = true_ergotropy_scalar(rho_super, H_annni)

# =======================
# Plot
# =======================
fig, ax = plt.subplots()

for n_shots in shot_list:
    med, low, up = compute_quartile_stats_from_array(
        results_by_shots[n_shots]
    )

    # Enforce positivity
    med = np.maximum(med, 0)
    low_band = np.maximum(med - low, 0)
    up_band  = np.maximum(med + up, 0)

    color = shot_color_map[n_shots]

    # Interquartile range
    ax.fill_between(
        x,
        low_band,
        up_band,
        color=color,
        alpha=0.25,
        linewidth=0,
    )

    # Median line
    ax.plot(
        x,
        med,
        marker="o",
        markersize=3,
        linewidth=1.2,
        color=color,
        label=rf"${n_shots:,}\ \mathrm{{shots}}$",
    )

# =======================
# True ergotropy line (darker than all curves)
# =======================
ax.axhline(
    erg_true,
    color="black",
    linestyle="--",
    linewidth=1.4,
    zorder=10,
    #label=r"$\mathcal{E}_{\mathrm{true}}$",
)

# Zero line
ax.axhline(0, color="gray", linewidth=0.6)

# Labels
#ax.set_xlabel(r"$|\mathcal{I}|$")
ax.set_xlabel(r"$K$")
ax.set_ylabel(r"$\mathcal{E}_{\mathrm{LB}}$")

# y axis range
ax.set_ylim(bottom=-0.2, top=erg_true + 0.2)

# Grid
ax.grid(True, linestyle="--", linewidth=0.5)

# Legend
ax.legend(frameon=False, loc="lower right")

plt.tight_layout()
plt.savefig(
    "probabilistic_ergotropy_ANNNI_open_boundaries.pdf",
    dpi=300,
    bbox_inches="tight",
)
plt.show()


/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_67400/1894510881.py:118: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


## Experimental ergotropy certification for a four-qubit GHZ state

We certify ergotropy lower bounds from experimental measurement data for a
four-qubit GHZ state under an XXZ Hamiltonian with parameters $J_1 = J_y = 1, \qquad \Delta = 0.5, \qquad B = 0.$

Pauli constraints are incorporated in the **fixed experimental order**
(without resampling). Each Pauli expectation value is estimated from
$2^{14}$ measurement shots.

Each point corresponds to a **single deterministic realization**.
The optimal unitary is updated only when it yields a strictly tighter
bound, ensuring **monotonicity** of the certified ergotropy curve.


Load experimental data

In [133]:
import numpy as np

# --- FILE 1: ideal GHZ state ---
state_ghz = np.loadtxt("states_ghz.txt", dtype=complex)
rho_ghz = np.outer(state_ghz, state_ghz.conj())

# --- FILE 2: measurement settings (0=X, 1=Y, 2=Z) ---
settings = np.loadtxt("strings_ghz.txt", dtype=int)

# --- FILE 3: outcome counts ---
counts = np.loadtxt("counts_ghz.txt", dtype=int)

num_settings = settings.shape[0]
assert counts.shape == (num_settings, 16)


Pauli operators and expectation values

In [134]:
from numpy.linalg import eigh
from functools import reduce
from numpy import kron

# single-qubit Pauli matrices
X = np.array([[0, 1], [1, 0]])
Y = np.array([[0, -1j], [1j, 0]])
Z = np.array([[1, 0], [0, -1]])
paulis = [X, Y, Z]

def pauli_operator_from_setting(setting_row):
    """Return the many-body Pauli operator for one measurement setting."""
    op = 1
    for s in setting_row:
        op = kron(op, paulis[s])
    return op

def bitstrings(n):
    return [np.array(list(np.binary_repr(i, width=n)), dtype=int)
            for i in range(2**n)]

bs = bitstrings(4)

def eigenvalue(b):
    """Eigenvalue for parity-type GHZ measurements."""
    return (-1) ** (b.sum())

# --- compute experimental expectation values ---
exp_vals = []

for k in range(num_settings):
    row = counts[k]
    shots = row.sum()
    probs = row / shots
    exp_val = sum(probs[i] * eigenvalue(bs[i]) for i in range(16))
    exp_vals.append(exp_val)

pauli_values = np.array(exp_vals)
pauli_ops = [pauli_operator_from_setting(s) for s in settings]


XXZ Hamiltonian (open boundaries)

In [136]:
L = 4

H_xxz = build_hamiltonian(
    L,
    J1=1.0,
    Jy=1.0,
    Delta=0.5,
    B=0.0,
    G=0.0
)


In [137]:
def true_ergotropy_scalar(rho, H):
    p, _ = eigh(rho)
    E, _ = eigh(H)

    p_sorted = np.sort(p)[::-1]
    E_sorted = np.sort(E)

    E_init = np.real(np.trace(rho @ H))
    E_passive = np.sum(p_sorted * E_sorted)

    return E_init - E_passive

erg_true = true_ergotropy_scalar(rho_ghz, H_xxz)
print(f"True ergotropy (GHZ–XXZ): {erg_true:.6f}")


True ergotropy (GHZ–XXZ): 2.139410


In [138]:
def sdp_most_mixed_state_relaxed_operators(L, pauli_ops, pauli_values, epsilon, solver=cp.SCS):
    """
    Operator-based relaxed SDP:
    - pauli_ops: list of (2^L × 2^L) numpy arrays
    - pauli_values: expectation values
    - epsilon: Hoeffding uncertainty width
    """
    rho = cp.Variable((2**L, 2**L), hermitian=True)

    # Minimize purity
    purity = cp.sum_squares(cp.abs(rho))
    objective = cp.Minimize(cp.real(purity))

    constraints = [
        cp.trace(rho) == 1,
        rho >> 0
    ]

    for Oi, oi in zip(pauli_ops, pauli_values):
        oi = cp.real(oi)
        constraints += [
            cp.real(cp.trace(rho @ Oi)) >= oi - epsilon,
            cp.real(cp.trace(rho @ Oi)) <= oi + epsilon
        ]

    try:
        problem = cp.Problem(objective, constraints)
        problem.solve(solver=solver, verbose=False)

        if problem.status not in ["optimal", "optimal_inaccurate"]:
            print(f"[SDP Warning] status: {problem.status}")
            return None

        return rho.value

    except Exception as e:
        print(f"[SDP Error] {e}")
        return None


In [139]:
def sdp_minimization_relaxed_operators(L, pauli_ops, pauli_values, U, H, epsilon, solver=cp.SCS):
    rho = cp.Variable((2**L, 2**L), hermitian=True)

    objective = cp.Minimize(
        cp.real(cp.trace((H - U.conj().T @ H @ U) @ rho))
    )

    constraints = [
        cp.trace(rho) == 1,
        rho >> 0
    ]

    for Oi, oi in zip(pauli_ops, pauli_values):
        oi = cp.real(oi)
        constraints += [
            cp.real(cp.trace(rho @ Oi)) >= oi - epsilon,
            cp.real(cp.trace(rho @ Oi)) <= oi + epsilon
        ]

    try:
        problem = cp.Problem(objective, constraints)
        problem.solve(solver=solver, verbose=False)

        if problem.status not in ["optimal", "optimal_inaccurate"]:
            print(f"[SDP Warning] status: {problem.status}")
            return np.inf, None

        return problem.value, rho.value

    except Exception as e:
        print(f"[SDP Error] {e}")
        return np.inf, None


**Deterministic certification with ordered constraints**

We sequentially incorporate Pauli constraints in the **experimental order**.
At each step:

1. We find a feasible state via a relaxed SDP.
2. We compute a candidate optimal unitary.
3. We accept the new unitary **only if it tightens the bound**.
4. Otherwise, we reuse the previous unitary.

This guarantees a **monotonic certified ergotropy curve**.


Ordered SDP with unitary reuse

In [141]:
Ns_list = np.arange(1, num_settings + 1)
n_shots = counts[0].sum()

medians = []
iqr_lower = []
iqr_upper = []

previous_lb = None
previous_U = None

for Ns in Ns_list:

    ops_subset  = pauli_ops[:Ns]
    vals_subset = pauli_values[:Ns]

    epsilon = compute_epsilon(Ns, n_shots)

    # 1) feasible state
    rho_feas = sdp_most_mixed_state_relaxed_operators(
    4, ops_subset, vals_subset, epsilon
)
    if rho_feas is None:
        medians.append(np.nan)
        iqr_lower.append(np.nan)
        iqr_upper.append(np.nan)
        continue

    # 2) candidate unitary
    U_candidate = optimal_unitary(rho_feas, H_xxz)

    # 3) candidate lower bound
    lb_candidate, _ = sdp_minimization_relaxed_operators(
    4, ops_subset, vals_subset, U_candidate, H_xxz, epsilon
)

    # 4) monotonicity check
    if previous_lb is not None and np.isfinite(lb_candidate):
        if lb_candidate < previous_lb:
        # use previous U and operators
            lb_used, _ = sdp_minimization_relaxed_operators(
        L, ops_subset, vals_subset, previous_U, H_xxz, epsilon
        )
        else:
            lb_used = lb_candidate
            previous_U = U_candidate
    else:
        lb_used = lb_candidate
        previous_U = U_candidate

    previous_lb = lb_used

    medians.append(lb_used)
    iqr_lower.append(0.0)
    iqr_upper.append(0.0)

medians = np.array(medians)


In [143]:
import matplotlib.pyplot as plt
import pylab

# ---- LaTeX-style figure ----
fig_width_pt = 246.0
inches_per_pt = 1.0 / 72.27
fig_width = fig_width_pt * inches_per_pt
fig_size = [fig_width, 0.75 * fig_width]

params = {
    'axes.labelsize': 10,
    'font.size': 9,
    'legend.fontsize': 9,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'text.usetex': True,
    'figure.figsize': fig_size
}
pylab.rcParams.update(params)

x = Ns_list
y = np.maximum(medians, 0)

fig, ax = plt.subplots()

ax.plot(
    x, y,
    marker='o',
    markersize=3,
    linewidth=1.2,
    color="#4C72B0",
    #label=r"$\mathcal{E}_{\mathrm{LB}}$"
)

# true ergotropy (darker than shot curves)
ax.axhline(
    erg_true,
    linestyle='--',
    linewidth=1.4,
    color="#4C72B0",
    #label=r"$\mathcal{E}$"
)

ax.axhline(0, color='gray', linewidth=0.6)

#ax.set_xlabel(r"$|\mathcal{I}|$")
ax.set_xlabel(r"$K$")
ax.set_ylabel(r"$\mathcal{E}_{\mathrm{LB}}$")

ax.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.grid(True, which='minor', linestyle=':', linewidth=0.2)

ax.legend(loc="lower right", frameon=False)
plt.tight_layout()

fname = "ergotropy_bounds_GHZ_xxz_experiment.pdf"
plt.savefig(fname, dpi=300, bbox_inches="tight")
plt.show()

print("Saved:", fname)


Saved: ergotropy_bounds_GHZ_xxz_experiment.pdf


/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_67400/2131381379.py:53: UserWarning: No artists with labels found to put in legend.  Note that artists whose label start with an underscore are ignored when legend() is called with no argument.
  ax.legend(loc="lower right", frameon=False)
/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_67400/2131381379.py:58: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()


## Ergotropy vs Mean Energy: Superposition of Extremal Eigenstates
This notebook computes lower bounds on the ergotropy of a four-qubit XXZ Hamiltonian system as a function of the **mean energy** of the state. 

We consider:
- **Blue curve**: our two-step protocol (SDP lower bound with optimal unitary).  
- **Red curve**: Hamiltonian-only bound (Canzio 2025).  
- **Green curve**: energy-diagonal (incoherent) ergotropy lower bound.  

The states are parametrized as superpositions of the **lowest- and highest-energy eigenstates**:
$$
|\psi(t)\rangle = \cos t |\epsilon_1\rangle + \sin t |\epsilon_d\rangle.
$$


In [220]:
# --- Imports and basic setup ---
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import eigh
from tqdm import tqdm
import itertools

# --- Define Pauli matrices ---
I = np.array([[1,0],[0,1]])
X = np.array([[0,1],[1,0]])
Y = np.array([[0,-1j],[1j,0]])
Z = np.array([[1,0],[0,-1]])
paulis_all = {'I': I, 'X': X, 'Y': Y, 'Z': Z}


**System definition: XXZ Hamiltonian (4 qubits)**
We use `build_hamiltonian` to construct the Hamiltonian:
$$
H = -J_1 \sum X_i X_{i+1} - J_y \sum Y_i Y_{i+1} - \Delta \sum Z_i Z_{i+1} 
$$
with open boundary conditions.


In [221]:
# --- System parameters ---
L = 4
J1 = 1.0
Jy = 1.0
Delta = 0.5
B = 0.0
G = 0.0

# --- Build Hamiltonian ---
H = build_hamiltonian(L, J1=J1, J2=0.0, Jy=Jy, Delta=Delta, B=B, G=G)

# --- Eigen-decomposition ---
eigs, vecs = eigh(H)
psi_min = vecs[:, 0]
psi_max = vecs[:, -1]


We only consider **nonzero Pauli terms in the Hamiltonian decomposition**. This reduces the number of measurements needed in the SDP.


In [222]:
def paulis_from_hamiltonian(H, L, tol=1e-9):
    """
    Returns a list of Pauli strings with nonzero coefficients in H.
    """
    paulis = ['I','X','Y','Z']
    dim = 2**L
    pauli_strings = []
    pauli_ops = []
    # iterate over all Pauli strings
    for p in itertools.product(paulis, repeat=L):
        s = ''.join(p)
        op = pauli_operator(s)  # use your existing function
        c = np.trace(H @ op).real / dim
        if abs(c) > tol:
            pauli_strings.append(s)
    return pauli_strings

paulis_H = paulis_from_hamiltonian(H, L)


**Compute lower bounds as a function of the superposition angle $t$**
- **Blue**: SDP two-step protocol (with `sdp_most_mixed_state_relaxed` + `sdp_minimization_relaxed`)  
- **Green**: Incoherent (energy-diagonal) ergotropy lower bound  
- **Red**: Hamiltonian-only theoretical bound (Canzio 2025)


**Build Hamiltonian-only theoretical curve**
- Uses `minErgPoints` and `minErg_array` 
- Red curve in plot.


In [223]:
def minErgPoints(hspectrum):
    """Compute the energy-constrained minimal ergotropy curve (vertices)."""
    hs = np.sort(np.asarray(hspectrum, dtype=float))
    d = len(hs)

    critE, critR = [], []

    for k in range(d):
        upperAvg = np.sum(hs[k:]) / (d - k)
        lowerAvg = np.sum(hs[:d - k]) / (d - k)
        critE.append(upperAvg)
        critR.append(upperAvg - lowerAvg)

    # convex lower envelope
    minE, minR = [critE[0]], [critR[0]]
    k = 0
    while k < d - 1:
        h = k + 1
        m = (critR[h] - minR[-1]) / (critE[h] - minE[-1])
        while h < d:
            testm = (critR[h] - minR[-1]) / (critE[h] - minE[-1])
            if testm <= m:
                m = testm
                k = h
            h += 1
        minE.append(critE[k])
        minR.append(critR[k])

    return np.array(minE), np.array(minR), np.array(critE), np.array(critR)

def minErg_array(E_array, minEnergies, minErgotropies):
    """Piecewise linear evaluation of minimal ergotropy from vertices."""
    E_array = np.atleast_1d(E_array)
    out = np.zeros_like(E_array, dtype=float)
    for i, E in enumerate(E_array):
        if E <= minEnergies[0]:
            out[i] = 0.0
        elif E >= minEnergies[-1]:
            out[i] = minErgotropies[-1]
        else:
            a = np.searchsorted(minEnergies, E) - 1
            a = max(0, min(a, len(minEnergies) - 2))
            E_a, E_b = minEnergies[a], minEnergies[a+1]
            erg_a, erg_b = minErgotropies[a], minErgotropies[a+1]
            t = (E - E_a) / (E_b - E_a)
            out[i] = (1 - t) * erg_a + t * erg_b
    return out


In [224]:
num_realizations = 10
t_values = np.linspace(0, np.pi/2, 12)

E_values = []
incoh_LBs = []
sdp_medians = []
sdp_lowers = []
sdp_uppers = []

for t in tqdm(t_values):
    # --- Superposition state ---
    psi = np.cos(t)*psi_min + np.sin(t)*psi_max
    psi = psi / np.linalg.norm(psi)
    rho = np.outer(psi, psi.conj())

    # --- Energy ---
    E = np.real(np.trace(rho @ H))
    E_values.append(E)

    # --- Incoherent lower bound ---
    overlaps = vecs.conj().T @ psi
    p = np.abs(overlaps)**2
    p[p<0] = 0.0
    p /= np.sum(p)
    p_desc = np.sort(p)[::-1]
    passive_energy = np.sum(p_desc * eigs)
    incoh_erg = max(E - passive_energy, 0.0)
    incoh_LBs.append(incoh_erg)

    # --- SDP realizations ---
    realization_vals = []
    for r in range(num_realizations):
        vals = pauli_expectations(rho, paulis_H)
        rho_feas = sdp_most_mixed_state(L, paulis_H, vals)
        if rho_feas is None:
            continue
        U_opt = optimal_unitary(rho_feas, H)
        min_val_2, _ = sdp_minimization(L, paulis_H, vals, U_opt, H)
        if np.isfinite(min_val_2):
            realization_vals.append(min_val_2)

    # --- Compute median + IQR ---
    if len(realization_vals) < 2:
        sdp_medians.append(np.nan)
        sdp_lowers.append(np.nan)
        sdp_uppers.append(np.nan)
    else:
        arr = np.array(realization_vals)
        med = np.median(arr)
        q25, q75 = np.percentile(arr, [25, 75])
        sdp_medians.append(med)
        sdp_lowers.append(med - q25)
        sdp_uppers.append(q75 - med)

# --- Convert to arrays and clip negatives ---
E_values = np.array(E_values)
incoh_LBs = np.array(incoh_LBs)
sdp_medians = np.array(sdp_medians)
sdp_lowers = np.array(sdp_lowers)
sdp_uppers = np.array(sdp_uppers)

sdp_medians_clipped = np.maximum(sdp_medians, 0)
sdp_lower_clipped = np.maximum(sdp_medians - sdp_lowers, 0)
sdp_upper_clipped = np.maximum(sdp_medians + sdp_uppers, 0)


100%|██████████| 12/12 [00:17<00:00,  1.43s/it]


In [225]:
minE_th, minR_th, critE_th, critR_th = minErgPoints(eigs)
E_grid = np.linspace(eigs.min()-0.1, eigs.max()+0.1, 800)
theory_curve = minErg_array(E_grid, minE_th, minR_th)


In [226]:
import pylab
import matplotlib as mpl

# --- PRL-style figure setup ---
fig_width_pt = 246.0
inches_per_pt = 1.0 / 72.27
fig_width = fig_width_pt * inches_per_pt
fig_size = [fig_width, 0.75*fig_width]

params = {
    'backend': 'ps',
    'axes.labelsize': 10,
    'font.size': 9,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'text.usetex': True,
    'figure.figsize': fig_size
}
pylab.rcParams.update(params)
mpl.rcParams.update({
    'text.usetex': True,
    'text.latex.preamble': r'\usepackage{amsmath,amssymb}'
})

fig, ax = plt.subplots()

# --- x-axis padding / limits ---
x_min, x_max = E_grid.min(), E_grid.max()
padding = 0.03*(x_max - x_min)
ax.set_xlim([x_min - padding, x_max + padding])
ax.set_ylim(bottom=-0.1, top=10)

# --- theoretical curve (red) ---
ax.plot(
    E_grid,
    theory_curve,
    color="#EE4B2B",
    linestyle='-',
    linewidth=1.2
)

# --- SDP median + IQR (blue) ---
color_sdp = "#4C72B0"
ax.fill_between(
    E_values,
    sdp_lower_clipped,
    sdp_upper_clipped,
    color=color_sdp,
    alpha=0.25
)
ax.plot(
    E_values,
    sdp_medians_clipped,
    marker='o',
    linestyle='-',
    color=color_sdp,
    linewidth=1.2,
    markersize=3
)

# --- Incoherent LB (green) ---
ax.plot(
    E_values,
    incoh_LBs,
    color="#55A868",
    linestyle='-',
    linewidth=1.2,
    markersize=3
)

# --- Zero line ---
ax.axhline(0, color='gray', linestyle='-', linewidth=0.6)

# --- Labels ---
ax.set_xlabel(r"$E$")
#ax.set_ylabel(r"$\mathcal{E}_{\mathrm{LB}}$")
ax.set_ylabel(r"Ergotropy lower bound")

# --- Inline blue label near SDP curve ---
idx = np.nanargmax(sdp_medians_clipped)
# --- Inline blue label (fixed position) ---
ax.text(
    -0.3, 2.0,
    r"$\mathcal{E}_{\mathrm{LB}}$",
    color=color_sdp,
    fontsize=9
)

# --- Grid ---
ax.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.grid(True, which='minor', linestyle=':', linewidth=0.2)

plt.tight_layout()

# --- Save figure ---
fname = f"correct_signs_comparison_theory_vs_SDP_hamiltonian_paulis_L{L}_canzio.pdf"
plt.savefig(fname, dpi=300, bbox_inches='tight')
print("Saved:", fname)

plt.show()


Saved: correct_signs_comparison_theory_vs_SDP_hamiltonian_paulis_L4_canzio.pdf


/var/folders/bl/2qzs5zlx0_b0dtwkqzfj3fznfc_zz4/T/ipykernel_67400/3450316676.py:100: UserWarning: FigureCanvasPS is non-interactive, and thus cannot be shown
  plt.show()
